In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Threshold Selection for High-Stakes Decisions\n",
    "\n",
    "Your model outputs probabilities. You need to make binary decisions.\n",
    "\n",
    "**The naive approach**: threshold at 0.5  \n",
    "**The reality**: The right threshold depends entirely on your use case.\n",
    "\n",
    "In cancer screening:\n",
    "- **Too aggressive** (low threshold): Many false positives → unnecessary biopsies\n",
    "- **Too conservative** (high threshold): Missed cancers → delayed treatment"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.datasets import make_classification\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.metrics import roc_curve, confusion_matrix\n",
    "\n",
    "import sys\n",
    "sys.path.append('..')\n",
    "from evaluation import ThresholdOptimizer\n",
    "\n",
    "sns.set_style(\"whitegrid\")\n",
    "np.random.seed(42)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## The Scenario\n",
    "\n",
    "We have a breast cancer screening model with ~5% prevalence."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "X, y = make_classification(\n",
    "    n_samples=10000,\n",
    "    n_features=20,\n",
    "    n_informative=10,\n",
    "    weights=[0.95, 0.05],\n",
    "    random_state=42,\n",
    "    flip_y=0.05\n",
    ")\n",
    "\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)\n",
    "\n",
    "print(f\"Training samples: {len(X_train)}\")\n",
    "print(f\"Test samples: {len(X_test)}\")\n",
    "print(f\"Positive rate: {y_test.mean():.1%}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "model = LogisticRegression(random_state=42, max_iter=1000)\n",
    "model.fit(X_train, y_train)\n",
    "y_prob = model.predict_proba(X_test)[:, 1]\n",
    "\n",
    "print(f\"Probability range: [{y_prob.min():.3f}, {y_prob.max():.3f}]\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## The Default: Threshold = 0.5"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "y_pred_default = (y_prob >= 0.5).astype(int)\n",
    "tn, fp, fn, tp = confusion_matrix(y_test, y_pred_default).ravel()\n",
    "\n",
    "sensitivity = tp / (tp + fn)\n",
    "specificity = tn / (tn + fp)\n",
    "\n",
    "print(\"Default Threshold = 0.5\")\n",
    "print(\"=\" * 40)\n",
    "print(f\"True Positives:  {tp}\")\n",
    "print(f\"False Negatives: {fn}  <- MISSED CANCERS\")\n",
    "print(f\"False Positives: {fp}\")\n",
    "print(f\"True Negatives:  {tn}\")\n",
    "print()\n",
    "print(f\"Sensitivity: {sensitivity:.1%}\")\n",
    "print(f\"Specificity: {specificity:.1%}\")\n",
    "print()\n",
    "print(f\"We're missing {fn} cancers out of {tp + fn} total!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Finding Better Thresholds"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "optimizer = ThresholdOptimizer()\n",
    "\n",
    "results = {\n",
    "    'Default (0.5)': 0.5,\n",
    "    'Youden J': optimizer.optimize_youden(y_test, y_prob).optimal_threshold,\n",
    "    'Sensitivity >= 95%': optimizer.optimize_sensitivity(y_test, y_prob, min_sensitivity=0.95).optimal_threshold,\n",
    "    'Sensitivity >= 90%': optimizer.optimize_sensitivity(y_test, y_prob, min_sensitivity=0.90).optimal_threshold,\n",
    "    'Max F1': optimizer.optimize_f1(y_test, y_prob).optimal_threshold,\n",
    "}\n",
    "\n",
    "for name, thresh in results.items():\n",
    "    print(f\"{name}: {thresh:.3f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Comparing Operating Points"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "comparison = []\n",
    "\n",
    "for name, thresh in results.items():\n",
    "    y_pred = (y_prob >= thresh).astype(int)\n",
    "    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()\n",
    "    \n",
    "    comparison.append({\n",
    "        'Strategy': name,\n",
    "        'Threshold': f\"{thresh:.3f}\",\n",
    "        'Sensitivity': f\"{tp / (tp + fn):.1%}\",\n",
    "        'Specificity': f\"{tn / (tn + fp):.1%}\",\n",
    "        'Missed Cancers': fn,\n",
    "        'Unnecessary Biopsies': fp\n",
    "    })\n",
    "\n",
    "comparison_df = pd.DataFrame(comparison)\n",
    "print(comparison_df.to_string(index=False))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Visualizing the Trade-off"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fpr, tpr, thresholds = roc_curve(y_test, y_prob)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 8))\n",
    "\n",
    "ax.plot(fpr, tpr, 'b-', linewidth=2, label='Model')\n",
    "ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')\n",
    "\n",
    "colors = plt.cm.Set1(np.linspace(0, 1, len(results)))\n",
    "for (name, thresh), color in zip(results.items(), colors):\n",
    "    idx = np.argmin(np.abs(thresholds - thresh))\n",
    "    ax.scatter(fpr[idx], tpr[idx], s=150, c=[color], marker='o', \n",
    "               edgecolors='black', linewidth=2, zorder=5)\n",
    "    ax.annotate(f\"{name}\\n(t={thresh:.2f})\", \n",
    "                xy=(fpr[idx], tpr[idx]),\n",
    "                xytext=(fpr[idx] + 0.05, tpr[idx] - 0.05),\n",
    "                fontsize=9)\n",
    "\n",
    "ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)\n",
    "ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)\n",
    "ax.set_title('ROC Curve with Operating Points', fontsize=14)\n",
    "ax.legend(loc='lower right')\n",
    "ax.grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## The Bottom Line\n",
    "\n",
    "**For cancer screening**, we'd likely choose **Sensitivity >= 95%** as our operating point."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "recommended = optimizer.optimize_sensitivity(y_test, y_prob, min_sensitivity=0.95)\n",
    "\n",
    "print(\"\\n\" + \"=\" * 60)\n",
    "print(\" RECOMMENDED OPERATING POINT FOR CANCER SCREENING\")\n",
    "print(\"=\" * 60)\n",
    "print(recommended)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}